In [1]:
import gc
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import CountVectorizer

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

import pandas as pd #pour la manipulation de données
import numpy as np
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte
from spacy.lang.fr.stop_words import STOP_WORDS as spacy_stopwords

from bertopic import BERTopic #pour la modélisation de sujets
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic
from sentence_transformers import SentenceTransformer #pour les embeddings de phrase

from umap import UMAP #pour la réduction de dimensionnalité
from hdbscan import HDBSCAN #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from sklearn.metrics import silhouette_score

from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel


df=pd.read_csv(Path("..")/"data" /"2_processed"/"03_corpus_lematise.csv", encoding="utf-8")

embedding_model = SentenceTransformer(
    "dangvantuan/sentence-camembert-base"
)

print("Génération des embeddings sémantiques...")
embeddings = embedding_model.encode(df['texte'].tolist(),batch_size=64, show_progress_bar=True)

# Documents utilisés par BERTopic
documents = (
    df["phrases_lemm"]
    .fillna("")
    .astype(str)
    .tolist()
)

# Textes tokenisés nécessaires au calcul de C_v
texts_tokenises = [
    document.split()
    for document in documents
]

dictionary = Dictionary(texts_tokenises)

# Vérifications élémentaires
embeddings_array = np.asarray(embeddings)

assert len(documents) == len(embeddings_array), (
    "Le nombre de documents ne correspond pas au nombre d'embeddings."
)

print("Documents :", len(documents))
print("Dimensions des embeddings :", embeddings_array.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Génération des embeddings sémantiques...


Batches:   0%|          | 0/84 [00:00<?, ?it/s]

Documents : 5327
Dimensions des embeddings : (5327, 768)


In [2]:
def calculer_coherence_cv(
    topic_model,
    labels,
    texts_tokenises,
    dictionary,
    top_n_words=10
):
    """
    Calcule la cohérence C_v des topics BERTopic,
    en ignorant le topic -1 correspondant aux outliers.
    """

    topic_ids = sorted(
        topic_id
        for topic_id in np.unique(labels)
        if topic_id != -1
    )

    topic_words = []

    for topic_id in topic_ids:
        representation = topic_model.get_topic(int(topic_id))

        if not representation:
            continue

        words = [
            word
            for word, _ in representation[:top_n_words]
        ]

        if words:
            topic_words.append(words)

    if len(topic_words) < 2:
        return np.nan

    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=texts_tokenises,
        dictionary=dictionary,
        coherence="c_v"
    )

    return coherence_model.get_coherence()

In [3]:
def evaluer_modele(
    topic_model,
    topics,
    texts_tokenises,
    dictionary,
    silhouette_sample_size=2000
):
    labels = np.asarray(topics)

    # Espace réellement utilisé par HDBSCAN
    embeddings_umap = np.asarray(
        topic_model.umap_model.embedding_
    )

    # Exclusion des outliers HDBSCAN
    masque_valide = labels != -1

    labels_valides = labels[masque_valide]
    embeddings_valides = embeddings_umap[masque_valide]

    topic_ids = np.unique(labels_valides)
    nombre_topics = len(topic_ids)

    taux_outliers = np.mean(labels == -1)

    # Distribution des tailles
    if nombre_topics > 0:
        tailles = pd.Series(labels_valides).value_counts()

        taille_min = int(tailles.min())
        taille_mediane = float(tailles.median())
        taille_max = int(tailles.max())
    else:
        taille_min = np.nan
        taille_mediane = np.nan
        taille_max = np.nan

    # La silhouette nécessite au moins deux clusters
    silhouette = np.nan

    if (
        nombre_topics >= 2
        and len(labels_valides) > nombre_topics
    ):
        if len(labels_valides) > silhouette_sample_size:
            silhouette = silhouette_score(
                embeddings_valides,
                labels_valides,
                metric="euclidean",
                sample_size=silhouette_sample_size,
                random_state=42
            )
        else:
            silhouette = silhouette_score(
                embeddings_valides,
                labels_valides,
                metric="euclidean"
            )

    coherence_cv = calculer_coherence_cv(
        topic_model=topic_model,
        labels=labels,
        texts_tokenises=texts_tokenises,
        dictionary=dictionary,
        top_n_words=10
    )

    return {
        "n_topics": nombre_topics,
        "outlier_rate": taux_outliers,
        "silhouette_umap": silhouette,
        "coherence_cv": coherence_cv,
        "min_topic_size_observed": taille_min,
        "median_topic_size": taille_mediane,
        "max_topic_size_observed": taille_max
    }

In [ ]:
param_grid = {
    "n_neighbors": [10, 25, 50],
    "n_components": [5, 10],
    "min_cluster_size": [20, 40, 80, 120],
    "min_samples": [1, 5, 10]
}

configurations = list(ParameterGrid(param_grid))

print("Nombre de configurations :", len(configurations))

Nombre de configurations : 54


In [23]:
resultats = []

for numero, params in enumerate(configurations, start=1):

    print(
        f"\nConfiguration {numero}/{len(configurations)} : "
        f"{params}"
    )

    debut = time.perf_counter()

    try:
        umap_model = UMAP(
            n_neighbors=params["n_neighbors"],
            n_components=params["n_components"],
            min_dist=0.0,
            metric="cosine",
            random_state=42,
            low_memory=True
        )

        hdbscan_model = HDBSCAN(
            min_cluster_size=params["min_cluster_size"],
            min_samples=params["min_samples"],
            metric="euclidean",
            cluster_selection_method="eom",
            prediction_data=True,
            core_dist_n_jobs=-1
        )

        # Une nouvelle instance à chaque configuration
        vectorizer_model = CountVectorizer(
            min_df=2,
            max_df=0.8,
            ngram_range=(1, 2)
        )

        ctfidf_model = ClassTfidfTransformer(
            reduce_frequent_words=True,
            bm25_weighting=True
        )

        topic_model_test = BERTopic(
            language="french",
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            ctfidf_model=ctfidf_model,
            nr_topics=None,
            calculate_probabilities=False,
            verbose=False
        )

        topics_test, _ = topic_model_test.fit_transform(
            documents,
            embeddings=embeddings_array
        )

        metriques = evaluer_modele(
            topic_model=topic_model_test,
            topics=topics_test,
            texts_tokenises=texts_tokenises,
            dictionary=dictionary,
            silhouette_sample_size=2000
        )

        duree = time.perf_counter() - debut

        ligne = {
            **params,
            **metriques,
            "duration_seconds": duree,
            "status": "ok",
            "error": None
        }

        print(
            f"Topics={metriques['n_topics']} | "
            f"Outliers={metriques['outlier_rate']:.1%} | "
            f"Silhouette={metriques['silhouette_umap']:.3f} | "
            f"C_v={metriques['coherence_cv']:.3f} | "
            f"Temps={duree:.1f}s"
        )

    except Exception as erreur:
        duree = time.perf_counter() - debut

        ligne = {
            **params,
            "n_topics": np.nan,
            "outlier_rate": np.nan,
            "silhouette_umap": np.nan,
            "coherence_cv": np.nan,
            "min_topic_size_observed": np.nan,
            "median_topic_size": np.nan,
            "max_topic_size_observed": np.nan,
            "duration_seconds": duree,
            "status": "error",
            "error": str(erreur)
        }

        print("Erreur :", erreur)

    resultats.append(ligne)

    # Sauvegarde progressive en cas d'interruption
    resultats_df = pd.DataFrame(resultats)

    resultats_df.to_csv(
        "resultats_grille_bertopic.csv",
        index=False
    )

    del topic_model_test
    gc.collect()


Configuration 1/54 : {'min_cluster_size': 30, 'min_samples': 5, 'n_components': 5, 'n_neighbors': 15}
Topics=22 | Outliers=55.8% | Silhouette=0.362 | C_v=0.426 | Temps=10.3s

Configuration 2/54 : {'min_cluster_size': 30, 'min_samples': 5, 'n_components': 5, 'n_neighbors': 30}
Topics=19 | Outliers=62.1% | Silhouette=0.391 | C_v=0.435 | Temps=11.8s

Configuration 3/54 : {'min_cluster_size': 30, 'min_samples': 5, 'n_components': 5, 'n_neighbors': 50}
Topics=15 | Outliers=67.5% | Silhouette=0.378 | C_v=0.416 | Temps=13.5s

Configuration 4/54 : {'min_cluster_size': 30, 'min_samples': 5, 'n_components': 10, 'n_neighbors': 15}
Topics=18 | Outliers=48.6% | Silhouette=0.297 | C_v=0.395 | Temps=10.5s

Configuration 5/54 : {'min_cluster_size': 30, 'min_samples': 5, 'n_components': 10, 'n_neighbors': 30}
Topics=21 | Outliers=65.1% | Silhouette=0.399 | C_v=0.476 | Temps=12.3s

Configuration 6/54 : {'min_cluster_size': 30, 'min_samples': 5, 'n_components': 10, 'n_neighbors': 50}
Topics=15 | Outlier

In [25]:
resultats_df = pd.DataFrame(resultats)

resultats_valides = (
    resultats_df
    .query("status == 'ok'")
    .sort_values(
        by=["coherence_cv", "silhouette_umap"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

colonnes_affichage = [
    "n_neighbors",
    "n_components",
    "min_cluster_size",
    "min_samples",
    "n_topics",
    "outlier_rate",
    "silhouette_umap",
    "coherence_cv",
    "min_topic_size_observed",
    "median_topic_size",
    "max_topic_size_observed",
    "duration_seconds"
]

display(
    resultats_valides[colonnes_affichage]
    .style.format({
        "outlier_rate": "{:.1%}",
        "silhouette_umap": "{:.3f}",
        "coherence_cv": "{:.3f}",
        "median_topic_size": "{:.1f}",
        "duration_seconds": "{:.1f}"
    })
)

,n_neighbors,n_components,min_cluster_size,min_samples,n_topics,outlier_rate,silhouette_umap,coherence_cv,min_topic_size_observed,median_topic_size,max_topic_size_observed,duration_seconds
0,15,10,50,10,8,57.0%,0.267,0.529,64,168.0,1023,10.6
1,15,5,30,15,14,67.8%,0.326,0.516,35,65.0,516,10.3
2,30,10,30,15,12,70.4%,0.436,0.493,30,84.5,460,12.4
3,50,10,50,10,6,70.2%,0.489,0.487,108,245.5,478,14.1
4,15,10,50,15,9,55.0%,0.270,0.486,55,82.0,1034,10.7
5,15,10,30,15,15,64.4%,0.450,0.479,31,71.0,541,10.5
6,30,10,30,5,21,65.1%,0.399,0.476,31,61.0,291,12.3
7,30,5,30,15,6,54.6%,0.106,0.475,58,156.0,1630,11.9
8,30,5,50,15,6,54.6%,0.106,0.475,58,156.0,1630,12.0
9,30,10,50,5,9,60.9%,0.334,0.470,69,142.0,814,12.5


In [41]:
selection = resultats_valides[(resultats_valides["outlier_rate"] <= 0.50)
]

selection = selection.sort_values(
    ["coherence_cv", "silhouette_umap"],
    ascending=False
)

display(selection[colonnes_affichage].head(10))

,n_neighbors,n_components,min_cluster_size,min_samples,n_topics,outlier_rate,silhouette_umap,coherence_cv,min_topic_size_observed,median_topic_size,max_topic_size_observed,duration_seconds
30,15,10,80,5,8,0.498217,0.309457,0.418305,91,219.5,860,10.666900
38,30,5,80,15,4,0.484701,0.292381,0.399212,134,378.5,1854,12.082031
39,15,10,30,5,18,0.485827,0.297206,0.395202,30,58.5,860,10.471017
46,15,5,80,5,7,0.497090,0.346903,0.377737,164,215.0,903,10.323541


In [42]:
meilleure_ligne = selection.iloc[0]

best_params = {
    "n_neighbors": int(meilleure_ligne["n_neighbors"]),
    "n_components": int(meilleure_ligne["n_components"]),
    "min_cluster_size": int(
        meilleure_ligne["min_cluster_size"]
    ),
    "min_samples": int(meilleure_ligne["min_samples"])
}

print(best_params)

{'n_neighbors': 15, 'n_components': 10, 'min_cluster_size': 80, 'min_samples': 5}


In [31]:
best_umap_model = UMAP(
    n_neighbors=best_params["n_neighbors"],
    n_components=best_params["n_components"],
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

best_hdbscan_model = HDBSCAN(
    min_cluster_size=best_params["min_cluster_size"],
    min_samples=best_params["min_samples"],
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

best_topic_model = BERTopic(
    language="french",
    umap_model=best_umap_model,
    hdbscan_model=best_hdbscan_model,
    vectorizer_model=CountVectorizer(
        min_df=2,
        max_df=0.8,
        ngram_range=(1, 2)
    ),
    ctfidf_model=ClassTfidfTransformer(
        reduce_frequent_words=True,
        bm25_weighting=True
    ),
    nr_topics=None,
    calculate_probabilities=False,
    verbose=True
)

best_topics_raw, _ = best_topic_model.fit_transform(
    documents,
    embeddings=embeddings_array
)

2026-07-27 16:09:41,410 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-27 16:09:46,002 - BERTopic - Dimensionality - Completed ✓
2026-07-27 16:09:46,002 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-27 16:09:46,105 - BERTopic - Cluster - Completed ✓
2026-07-27 16:09:46,107 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-27 16:09:46,903 - BERTopic - Representation - Completed ✓


In [43]:

topics_reduits = best_topic_model.topics_

best_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,1278,0_porte_cœur_maison_pauvre,"[porte, cœur, maison, pauvre, fois, soir, face...",[gros peine mort sûr intéresser gros larme mam...
1,1,954,1_haut_noir_long_porte,"[haut, noir, long, porte, fond, maison, blanc,...",[population mange balustrade curieux colossal ...
2,2,650,2_frère_franc_affaire_maison,"[frère, franc, affaire, maison, soir, dernier,...",[fils maître vocation métier manuel fondateur ...
3,3,639,3_dame_franc_porte_soir,"[dame, franc, porte, soir, maison, fois, gros,...",[immoralité remarque faute raisonnable partage...
4,4,612,4_cœur_amour_mort_chambre,"[cœur, amour, mort, chambre, fond, soir, fois,...",[matinée entier temps rue cheveu roux nuque pe...
5,5,579,5_chambre_porte_lit_maison,"[chambre, porte, lit, maison, face, fenêtre, f...",[matin frissonnante camisole vif fenêtre trave...
6,6,342,6_peuple_travail_nouveau_dernier,"[peuple, travail, nouveau, dernier, bonheur, œ...",[monument orgueil domination science nom idéal...
7,7,273,7_face_pied_mort_sang,"[face, pied, mort, sang, fois, gros, fond, cœu...",[mort lent convulsion corde solide immobilité ...


In [44]:
topics_sans_outliers = best_topic_model.reduce_outliers(
    documents,
    topics_reduits,
    strategy="embeddings",
    embeddings=embeddings_array,
    threshold=0.15
)

best_topic_model.update_topics(
    documents,
    topics=topics_sans_outliers)


ValueError: No outliers to reduce.

In [45]:

best_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,1278,0_porte_cœur_maison_pauvre,"[porte, cœur, maison, pauvre, fois, soir, face...",[gros peine mort sûr intéresser gros larme mam...
1,1,954,1_haut_noir_long_porte,"[haut, noir, long, porte, fond, maison, blanc,...",[population mange balustrade curieux colossal ...
2,2,650,2_frère_franc_affaire_maison,"[frère, franc, affaire, maison, soir, dernier,...",[fils maître vocation métier manuel fondateur ...
3,3,639,3_dame_franc_porte_soir,"[dame, franc, porte, soir, maison, fois, gros,...",[immoralité remarque faute raisonnable partage...
4,4,612,4_cœur_amour_mort_chambre,"[cœur, amour, mort, chambre, fond, soir, fois,...",[matinée entier temps rue cheveu roux nuque pe...
5,5,579,5_chambre_porte_lit_maison,"[chambre, porte, lit, maison, face, fenêtre, f...",[matin frissonnante camisole vif fenêtre trave...
6,6,342,6_peuple_travail_nouveau_dernier,"[peuple, travail, nouveau, dernier, bonheur, œ...",[monument orgueil domination science nom idéal...
7,7,273,7_face_pied_mort_sang,"[face, pied, mort, sang, fois, gros, fond, cœu...",[mort lent convulsion corde solide immobilité ...
